`LangGraph` 构建的计算图通常会和 `LLM` 结合使用，而 `LLM` 在运行过程中通常需要维护一组消息列表。为了提升开发效率，`LangGraph` 官方提供了一个预定义状态类型：`langgraph.graph.message.MessagesState`。

In [3]:
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState
from langchain.messages import HumanMessage
from dotenv import load_dotenv
from rich import print as rprint

load_dotenv(override=True)

deepseek_model = ChatDeepSeek(
    model="deepseek-flash",
    #api_key=os.getenv()
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    }
)

class OverAllState(MessagesState):
    username: str
    output: str

def node_a(state: OverAllState) -> OverAllState:
    return {
        "messages": [HumanMessage("你好，我是 " + state["username"])]
    }

def llm_node(state: OverAllState) -> OverAllState:
    res = deepseek_model.invoke(state["messages"])
    return {
        "messages": [res],
        "output": res.content
    }

builder = StateGraph(
    state_schema=OverAllState,
)

builder.add_node("node_a", node_a)
builder.add_node("llm_node", llm_node)

builder.add_edge(START, "node_a")
builder.add_edge("node_a", "llm_node")
builder.add_edge("llm_node", END)

graph = builder.compile()
resp = graph.invoke({"username": "小黄"})
rprint(resp)

{
    'messages': [
        HumanMessage(
            content='你好，我是 小黄',
            additional_kwargs={},
            response_metadata={},
            id='5f06149b-5ec1-45e8-9157-73150cc6b71a'
        ),
        AIMessage(
            content='你好，小黄！很高兴认识你。😊\n\n有什么我可以帮你的吗？无论是解答问题、聊天，还是需要一些建议，
我都在这里。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 33,
                    'prompt_tokens': 10,
                    'total_tokens': 43,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0
                    },
                    'prompt_cache_hit_tokens': 0,
                    'prompt_cache_miss_tokens': 10
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'e72d3c7f663d3e1383b00b3884cb4501',
                'id': 'cd2f6211-7674-4a5e-9598-706ab08a5ddf',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--01a14936-c49f-7980-944f-3aa454652214-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 10,
                'output_tokens': 33,
                'total_tokens': 43,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        )
    ],
    'username': '小黄',
    'output': 
'你好，小黄！很高兴认识你。😊\n\n有什么我可以帮你的吗？无论是解答问题、聊天，还是需要一些建议，我都在这里。'
}